Para emepezar, poner el jupyter en vs code con el kernel de python

Si no os sale, ejecutad estos tres comandos a la altura del path T20
- python3 -m venv .venv
- source .venv/bin/activate
- python -m pip install ipykernel

Sección	Contenido	Responsable
0. Introducción	Objetivo, integrantes y librerías	JORGE
1. Reparto	Apartado a): generación de participaciones	JORGE
2. Recuperación	Apartado b): interpolación de Newton	ARJO
3. Comprobación	Apartado c): ejemplo del profesor	Compañero INIESTA
4. Umbral	Apartado d): pruebas y explicación matemática	Compañero INIESTA
5. Aplicación AES	Apartado e): clave AES-256	FRAN
6. Lagrange	Apartado f): recuperación alternativa	FRAN (OPCIONAL)
7. Conclusiones	Resultados y comprobaciones finales	TODOS

Cada sección tendrá una explicación breve, su código Python comentado y los resultados de ejecución.
-
-
-
-
-
-
-
-
-
-
-
-


# Seguridad y Protección de Sistemas Informáticos
## Trabajo de Python — Esquema de Shamir

**Curso:** 2026/2027

**Integrantes:**
- Jorge Castillo Fuentes:
- Alfredo Iniesta Garcia:
- Francisco Pradillo Martinez:
- Alvaro Arjona Jimenez:

## Objetivo

Implementar el esquema de Shamir para compartir secretos,
permitiendo repartir un secreto entre varios participantes
y recuperarlo cuando se dispone de suficientes participaciones.

## Apartados

1. Reparto del secreto
2. Recuperación mediante interpolación de Newton
3. Comprobación con el ejemplo de los apuntes
4. Comprobación del umbral de seguridad
5. Aplicación con una clave AES de 256 bits
6. Interpolación de Lagrange (opcional)

## 1. Reparto del secreto

En este apartado se implementará una función que permita
dividir un secreto en n participaciones mediante el
esquema de Shamir.

Para recuperar el secreto se necesitarán al menos
t participaciones.

La función recibirá:
- s: secreto que queremos repartir.
- n: número de participantes.
- t: número mínimo de participaciones.
- bits: tamaño máximo del secreto en bits.

La función devolverá el número primo p y las
participaciones generadas.

In [ ]:
import secrets
import subprocess
from itertools import combinations
import time

## 1. Reparto del secreto

En este apartado implementamos el reparto de un secreto
mediante el esquema de Shamir.

La función recibe el secreto, el número de participantes,
el umbral mínimo para recuperarlo y el tamaño máximo
del secreto en bits.

Primero comprobamos los parámetros. Después generamos
un número primo y los coeficientes aleatorios del
polinomio. Por último, elegimos los puntos y calculamos
las participaciones de cada participante.

Todas las operaciones del polinomio se realizan módulo p.

In [ ]:
def validar_parametros(s, n, t, bits):
    # Comprobamos las condiciones del enunciado.
    if not all(type(v) is int for v in (s, n, t, bits)):
        raise TypeError("Los parámetros deben ser enteros")

    if n < 3:
        raise ValueError("Debe haber al menos 3 participantes")

    if t < 2 or t > n:
        raise ValueError("El umbral debe estar entre 2 y n")

    if bits < 1:
        raise ValueError("El tamaño en bits debe ser positivo")

    if s < 0 or s >= 2**bits:
        raise ValueError("El secreto no cabe en los bits indicados")


def generar_primo(bits, n):
    # El primo debe superar tanto a n como a cualquier
    # secreto que pueda representarse con esos bits.
    bits_primo = max(bits + 1, n.bit_length() + 1)

    resultado = subprocess.run(
        ["openssl", "prime", "-generate", "-bits", str(bits_primo)],
        capture_output=True,
        text=True,
        check=True
    )

    return int(resultado.stdout.strip())


def evaluar_polinomio(coeficientes, x, p):
    # Evaluamos el polinomio módulo p mediante Horner.
    resultado = 0

    for coeficiente in reversed(coeficientes):
        resultado = (resultado * x + coeficiente) % p

    return resultado


def repartir_secreto(s, n, t, bits):
    validar_parametros(s, n, t, bits)

    # Generamos el primo.
    p = generar_primo(bits, n)

    # El primer coeficiente es el secreto.
    # Los demás se generan aleatoriamente, incluido el cero.
    coeficientes = [s]
    for _ in range(t - 1):
        coeficientes.append(secrets.randbelow(p))

    # Elegimos n puntos diferentes y distintos de cero.
    puntos = set()

    while len(puntos) < n:
        x = secrets.randbelow(p - 1) + 1
        puntos.add(x)

    # Calculamos la participación correspondiente a cada x.
    participaciones = []

    for x in sorted(puntos):
        y = evaluar_polinomio(coeficientes, x, p)
        participaciones.append((x, y))

    return p, participaciones

In [ ]:
# Ejemplo con 5 participantes y umbral de 3.
s = 1234
n = 5
t = 3
bits = 16

p, participaciones = repartir_secreto(s, n, t, bits)

print("Secreto:", s)
print("Número primo:", p)
print("Participantes:", n)
print("Umbral:", t)

print("\nParticipaciones generadas:")
for i, (x, y) in enumerate(participaciones, start=1):
    print(f"Participante {i}: ({x}, {y})")

print("\nComprobaciones:")
print("Primo mayor que n:", p > n)
print("Primo mayor que todos los secretos posibles:", p >= 2**bits)
print("Número correcto de participaciones:", len(participaciones) == n)
print("Puntos distintos:", len({x for x, y in participaciones}) == n)
print("Puntos no nulos:", all(x != 0 for x, y in participaciones))

## 3. Comprobación con el ejemplo de los apuntes

En este apartado reproducimos el **ejemplo 8.6.1** de los apuntes del curso.
Utilizamos los parámetros fijos del ejemplo ($p = 1499$, $n = 10$, $t = 7$ y $s = 1132$),
junto con los coeficientes y puntos $x_i$ indicados en el texto.

Primero calculamos las 10 participaciones evaluando el polinomio en $\mathbb{Z}_p$
y verificamos que coinciden con las del ejemplo. Después comprobamos que,
tomando cualquier subconjunto de $t = 7$ participaciones, la función de
recuperación devuelve siempre el secreto original $s = 1132$.

In [ ]:
# Parámetros fijos del ejemplo 8.6.1 de los apuntes
p_ej = 1499
s_ej = 1132
t_ej = 7

# TODO: Sustituir por los valores exactos del ejemplo 8.6.1 de los apuntes
# coeficientes = [s, a_1, a_2, a_3, a_4, a_5, a_6]
coeficientes_ej = [s_ej, 0, 0, 0, 0, 0, 0]  # <-- Poner aquí a_1 ... a_6
puntos_x_ej = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]  # <-- Poner aquí x_1 ... x_10

# Calculamos las 10 participaciones reutilizando evaluar_polinomio de la sección 1
participaciones_ej = [
    (x, evaluar_polinomio(coeficientes_ej, x, p_ej))
    for x in puntos_x_ej
]

print("Participaciones obtenidas para el ejemplo 8.6.1:")
for i, (x, y) in enumerate(participaciones_ej, start=1):
    print(f"  Participante {i:2d}: ({x}, {y})")

# Comprobamos TODAS las combinaciones posibles de 7 participaciones entre 10 (120 en total)
todas_combinaciones = list(combinations(participaciones_ej, t_ej))
aciertos = 0

for subconjunto in todas_combinaciones:
    # Cambiar 'recuperar_secreto' por el nombre exacto de la función de la sección 2
    s_recuperado = recuperar_secreto(p_ej, list(subconjunto))
    if s_recuperado == s_ej:
        aciertos += 1

print(f"\nComprobación exhaustiva con t = {t_ej} participaciones:")
print(f"  Combinaciones probadas: {len(todas_combinaciones)}")
print(f"  Recuperaciones correctas (s = {s_ej}): {aciertos}/{len(todas_combinaciones)}")
print(f"  ¿Todas correctas?: {aciertos == len(todas_combinaciones)}")

## 4. Comprobación del umbral de seguridad

En este apartado verificamos experimentalmente la propiedad de umbral del esquema de Shamir:
- Con **$t$ participaciones** cualesquiera se recupera siempre el secreto original $s$.
- Con **$t - 1$ participaciones** (o menos) el valor interpolado en $0$ no corresponde,
  en general, al secreto real.

In [ ]:
import random

# Generamos un reparto aleatorio con parámetros de prueba (n=8, t=5)
s_umbral = 98765
n_umbral = 8
t_umbral = 5
bits_umbral = 20

p_umbral, participaciones_umbral = repartir_secreto(s_umbral, n_umbral, t_umbral, bits_umbral)

num_pruebas = 10
print(f"Secreto original: s = {s_umbral} | Primo p = {p_umbral} | Umbral t = {t_umbral}\n")

#Pruebas con exactamente t participaciones elegidas al azar
print(f"Pruebas con t = {t_umbral} participaciones")
aciertos_t = 0
for i in range(1, num_pruebas + 1):
    muestra = random.sample(participaciones_umbral, t_umbral)
    s_rec = recuperar_secreto(p_umbral, muestra)
    coincide = (s_rec == s_umbral)
    aciertos_t += coincide
    print(f"  Prueba {i:2d}: recuperado = {s_rec} | ¿Coincide con s? {coincide}")

#Pruebas con t - 1 participaciones elegidas al azar
print(f"\n--- Pruebas con t - 1 = {t_umbral - 1} participaciones ---")
aciertos_t_menos_1 = 0
for i in range(1, num_pruebas + 1):
    muestra = random.sample(participaciones_umbral, t_umbral - 1)
    s_rec = recuperar_secreto(p_umbral, muestra)
    coincide = (s_rec == s_umbral)
    aciertos_t_menos_1 += coincide
    print(f"  Prueba {i:2d}: recuperado = {s_rec} | ¿Coincide con s? {coincide}")

print(f"\nResumen:")
print(f"  Aciertos con t participaciones:     {aciertos_t}/{num_pruebas}")
print(f"  Aciertos con t - 1 participaciones: {aciertos_t_menos_1}/{num_pruebas}")

### ¿Por qué $t - 1$ participaciones no revelan ninguna información sobre $s$?

Supongamos que un atacante consigue reunir $t - 1$ participaciones:
$$\langle x_1, y_1 \rangle, \langle x_2, y_2 \rangle, \dots, \langle x_{t-1}, y_{t-1} \rangle$$

Como los puntos $x_i$ se eligen en $\mathbb{Z}_p^*$ y son distintos dos a dos,
ninguno de ellos es igual a $0$. Si el atacante intenta adivinar el secreto
probando cualquier valor candidato $s' \in \mathbb{Z}_p$, está añadiendo el punto
$(0, s')$ al conjunto anterior, formando así exactamente $t$ puntos con abscisas
distintas dos a dos.

Por el **teorema 8.6.2** (existencia y unicidad del polinomio interpolador en $\mathbb{Z}_p[x]$),
para cada posible valor $s' \in \mathbb{Z}_p$ existe **exactamente un único polinomio**
de grado menor o igual que $t - 1$ que pasa por las $t - 1$ participaciones conocidas
y por el punto $(0, s')$.

Dado que los coeficientes aleatorios $a_1, \dots, a_{t-1}$ se eligieron de forma
uniforme e independiente en $\mathbb{Z}_p$, cada uno de esos $p$ polinomios posibles
tiene exactamente la misma probabilidad de haber sido generado. En consecuencia,
todos los valores $s' \in \mathbb{Z}_p$ siguen siendo equiprobables a ojos del atacante:
conocer $t - 1$ participaciones no descarta ningún candidato ni aporta información
alguna sobre el secreto $s$ (secreto perfecto).